In [6]:
#setup - installation and imports

In [8]:
!pip install boto3 transformers torch numpy pandas

  Using cached boto3-1.43.2-py3-none-any.whl.metadata (6.5 kB)
  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached torch-2.11.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached botocore-1.43.2-py3-none-any.whl.metadata (5.5 kB)
  Using cached jmespath-1.1.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached s3transfer-0.17.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached huggingface_hub-1.13.0-py3-none-any.whl.metadata (14 kB)
  Using cached regex-2026.4.4-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached typer-0.25.1-py3-none-any.whl.metadata (15 kB)
  Using cached safetensors-0.7.0-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
  Using cached filelock-3.29.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached typing_extensions-4.15.0-py3

In [10]:
import os
import boto3
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from collections import Counter
import json
from datetime import datetime

BUCKET_NAME = "yegon-jay-hw2"
S3_PREFIX = "ml-1m"
LOCAL_DIR = "ml-1m"

In [ ]:
os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_SESSION_TOKEN"] = ""
os.environ["AWS_DEFAULT_REGION"] = "us-east-1"
s3 = boto3.client("s3")

In [12]:
# Task 1

In [13]:
def download_movielens(dest_dir="ml-1m"):
    url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
    zip_path = "ml-1m.zip"

    if os.path.exists(dest_dir) and os.listdir(dest_dir):
        print(f"'{dest_dir}' already exists locally, skipping download.")
        return dest_dir

    print("Downloading MovieLens 1M...")
    urllib.request.urlretrieve(url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(".")
    os.remove(zip_path)
    print(f"Extracted to '{dest_dir}/'")
    return dest_dir


def upload_dataset_to_s3(local_dir, bucket, prefix):
    for filename in os.listdir(local_dir):
        filepath = os.path.join(local_dir, filename)
        if os.path.isfile(filepath):
            s3_key = f"{prefix}/{filename}"
            print(f"Uploading {filename} -> s3://{bucket}/{s3_key}")
            s3.upload_file(filepath, bucket, s3_key)
    print("Upload complete.")

In [14]:
local_dir = download_movielens()
upload_dataset_to_s3(local_dir, BUCKET_NAME, S3_PREFIX)

'ml-1m' already exists locally, skipping download.
Uploading movies.dat -> s3://yegon-jay-hw2/ml-1m/movies.dat
Uploading ratings.dat -> s3://yegon-jay-hw2/ml-1m/ratings.dat
Uploading README -> s3://yegon-jay-hw2/ml-1m/README
Uploading users.dat -> s3://yegon-jay-hw2/ml-1m/users.dat
Upload complete.


In [15]:
def download_dataset_from_s3(bucket, prefix, local_dir="ml-1m"):
    os.makedirs(local_dir, exist_ok=True)
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix + "/")

    for obj in response.get("Contents", []):
        key = obj["Key"]
        filename = key.split("/")[-1]
        if filename:
            local_path = os.path.join(local_dir, filename)
            print(f"Downloading {key} -> {local_path}")
            s3.download_file(bucket, key, local_path)

    print("Done. Local files:", os.listdir(local_dir))

In [18]:
def dataset_exists_on_s3(bucket, prefix):
    response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix + "/", MaxKeys=1)
    exists = response.get("KeyCount", 0) > 0
    if exists:
        print(f"Dataset found on S3 at s3://{bucket}/{prefix}/")
    else:
        print(f"Dataset not found on S3.")
    return exists

In [23]:
def ensure_data_available(bucket, prefix, local_dir="ml-1m"):
    if not dataset_exists_on_s3(bucket, prefix):
        local_dir = download_movielens(local_dir)
        upload_dataset_to_s3(local_dir, bucket, prefix)
    elif not os.path.exists(local_dir) or not os.listdir(local_dir):
        print("Data on S3 but not local. Pulling from S3...")
        download_dataset_from_s3(bucket, prefix, local_dir)
    else:
        print("Data already available locally and on S3.")

ensure_data_available(BUCKET_NAME, S3_PREFIX)

Dataset found on S3 at s3://yegon-jay-hw2/ml-1m/
Data already available locally and on S3.


In [24]:
response = s3.list_objects_v2(Bucket="yegon-jay-hw2")
for obj in response.get("Contents", []):
    print(obj["Key"])

ml-1m/README
ml-1m/movies.dat
ml-1m/ratings.dat
ml-1m/users.dat


In [25]:
def load_movielens_data(data_dir="ml-1m"):
    movies = pd.read_csv(
        os.path.join(data_dir, "movies.dat"), sep="::", header=None,
        names=["MovieID", "Title", "Genres"], engine="python", encoding="latin-1"
    )
    ratings = pd.read_csv(
        os.path.join(data_dir, "ratings.dat"), sep="::", header=None,
        names=["UserID", "MovieID", "Rating", "Timestamp"], engine="python"
    )
    users = pd.read_csv(
        os.path.join(data_dir, "users.dat"), sep="::", header=None,
        names=["UserID", "Gender", "Age", "Occupation", "Zip"], engine="python"
    )
    print(f"Loaded {len(movies)} movies, {len(ratings)} ratings, {len(users)} users")
    return movies, ratings, users

movies, ratings, users = load_movielens_data()

Loaded 3883 movies, 1000209 ratings, 6040 users


In [26]:
movies, ratings, users = load_movielens_data()
print("Movies:")
print(movies.head())
print(f"\nRatings:")
print(ratings.head())
print(f"\nUsers:")
print(users.head())

Loaded 3883 movies, 1000209 ratings, 6040 users
Movies:
   MovieID                               Title                        Genres
0        1                    Toy Story (1995)   Animation|Children's|Comedy
1        2                      Jumanji (1995)  Adventure|Children's|Fantasy
2        3             Grumpier Old Men (1995)                Comedy|Romance
3        4            Waiting to Exhale (1995)                  Comedy|Drama
4        5  Father of the Bride Part II (1995)                        Comedy

Ratings:
   UserID  MovieID  Rating  Timestamp
0       1     1193       5  978300760
1       1      661       3  978302109
2       1      914       3  978301968
3       1     3408       4  978300275
4       1     2355       5  978824291

Users:
   UserID Gender  Age  Occupation    Zip
0       1      F    1          10  48067
1       2      M   56          16  70072
2       3      M   25          15  55117
3       4      M   45           7  02460
4       5      M   25          

In [27]:
#Task 2

In [28]:
def extract_year(title):
    try:
        return int(title.strip()[-5:-1])
    except:
        return None

movies["Year"] = movies["Title"].apply(extract_year)
print(f"Year range: {movies['Year'].min()} - {movies['Year'].max()}")
print(f"Movies with missing year: {movies['Year'].isna().sum()}")
movies[["Title", "Year"]].head(10)

Year range: 1919 - 2000
Movies with missing year: 0


,Title,Year
0,Toy Story (1995),1995
1,Jumanji (1995),1995
2,Grumpier Old Men (1995),1995
3,Waiting to Exhale (1995),1995
4,Father of the Bride Part II (1995),1995
5,Heat (1995),1995
6,Sabrina (1995),1995
7,Tom and Huck (1995),1995
8,Sudden Death (1995),1995
9,GoldenEye (1995),1995


In [29]:
movies_pre1980 = movies[movies["Year"] <= 1980].copy()
print(f"Movies released 1980 or earlier: {len(movies_pre1980)}")
movies_pre1980.head(10)

Movies released 1980 or earlier: 887


,MovieID,Title,Genres,Year
109,111,Taxi Driver (1976),Drama|Thriller,1976
152,154,Belle de jour (1967),Drama,1967
197,199,"Umbrellas of Cherbourg, The (Parapluies de Che...",Drama|Musical,1964
257,260,Star Wars: Episode IV - A New Hope (1977),Action|Adventure|Fantasy|Sci-Fi,1977
386,390,Faster Pussycat! Kill! Kill! (1965),Action|Comedy|Drama,1965
491,495,In the Realm of the Senses (Ai no corrida) (1976),Drama,1976
553,557,Mamma Roma (1962),Drama,1962
590,594,Snow White and the Seven Dwarfs (1937),Animation|Children's|Musical,1937
592,596,Pinocchio (1940),Animation|Children's,1940
595,599,"Wild Bunch, The (1969)",Western,1969


In [30]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()
print("BERT model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT model loaded.


In [31]:
def build_movie_text(row):
    title = row["Title"].rsplit("(", 1)[0].strip()
    genres = row["Genres"].replace("|", " ")
    year = row["Year"]
    return f"{title} {year} {genres}"

movies_pre1980["Text"] = movies_pre1980.apply(build_movie_text, axis=1)
movies_pre1980[["Title", "Text"]].head(10)

,Title,Text
109,Taxi Driver (1976),Taxi Driver 1976 Drama Thriller
152,Belle de jour (1967),Belle de jour 1967 Drama
197,"Umbrellas of Cherbourg, The (Parapluies de Che...","Umbrellas of Cherbourg, The (Parapluies de Che..."
257,Star Wars: Episode IV - A New Hope (1977),Star Wars: Episode IV - A New Hope 1977 Action...
386,Faster Pussycat! Kill! Kill! (1965),Faster Pussycat! Kill! Kill! 1965 Action Comed...
491,In the Realm of the Senses (Ai no corrida) (1976),In the Realm of the Senses (Ai no corrida) 197...
553,Mamma Roma (1962),Mamma Roma 1962 Drama
590,Snow White and the Seven Dwarfs (1937),Snow White and the Seven Dwarfs 1937 Animation...
592,Pinocchio (1940),Pinocchio 1940 Animation Children's
595,"Wild Bunch, The (1969)","Wild Bunch, The 1969 Western"


In [32]:
def generate_embeddings(texts, tokenizer, model, batch_size=32):
    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        encoded = tokenizer(batch, padding=True, truncation=True, max_length=64, return_tensors="pt")

        with torch.no_grad():
            outputs = model(**encoded)

        cls_embeddings = outputs.last_hidden_state[:, 0, :].numpy()
        all_embeddings.append(cls_embeddings)

        if (i // batch_size) % 5 == 0:
            print(f"Processed {min(i + batch_size, len(texts))}/{len(texts)} movies")

    return np.vstack(all_embeddings)

In [33]:
texts_pre1980 = movies_pre1980["Text"].tolist()
embeddings_pre1980 = generate_embeddings(texts_pre1980, tokenizer, model)
print(f"Embeddings shape: {embeddings_pre1980.shape}")

Processed 32/887 movies
Processed 192/887 movies
Processed 352/887 movies
Processed 512/887 movies
Processed 672/887 movies
Processed 832/887 movies
Embeddings shape: (887, 768)


In [34]:
os.makedirs("outputs", exist_ok=True)

np.save("outputs/embeddings_pre1980.npy", embeddings_pre1980)

movies_pre1980[["MovieID", "Title", "Genres", "Year", "Text"]].to_csv(
    "outputs/movies_pre1980.csv", index=False
)

print("Saved embeddings_pre1980.npy and movies_pre1980.csv locally.")

Saved embeddings_pre1980.npy and movies_pre1980.csv locally.


In [35]:
def upload_file_to_s3(local_path, bucket, s3_key):
    print(f"Uploading {local_path} -> s3://{bucket}/{s3_key}")
    s3.upload_file(local_path, bucket, s3_key)

upload_file_to_s3("outputs/embeddings_pre1980.npy", BUCKET_NAME, "outputs/embeddings_pre1980.npy")
upload_file_to_s3("outputs/movies_pre1980.csv", BUCKET_NAME, "outputs/movies_pre1980.csv")
print("Pre-1980 embeddings saved to S3.")

Uploading outputs/embeddings_pre1980.npy -> s3://yegon-jay-hw2/outputs/embeddings_pre1980.npy
Uploading outputs/movies_pre1980.csv -> s3://yegon-jay-hw2/outputs/movies_pre1980.csv
Pre-1980 embeddings saved to S3.


In [36]:
#if its in s3
response = s3.list_objects_v2(Bucket=BUCKET_NAME)
for obj in response.get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")

  ml-1m/README  (5577 bytes)
  ml-1m/movies.dat  (171308 bytes)
  ml-1m/ratings.dat  (24594131 bytes)
  ml-1m/users.dat  (134368 bytes)
  outputs/embeddings_pre1980.npy  (2724992 bytes)
  outputs/movies_pre1980.csv  (76262 bytes)


In [37]:
#Task 3

In [38]:
def cosine_similarity(a, b):
    dot = np.dot(a, b.T)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b, axis=1)
    return dot / (norm_a * norm_b + 1e-8)

In [39]:
#we recommend movies based on user embeddings
def recommend_movies(user_embedding, movie_embeddings, movie_df, seen_movie_ids=None, top_n=5):
    scores = cosine_similarity(user_embedding, movie_embeddings)
    movie_df = movie_df.copy()
    movie_df["Score"] = scores

    if seen_movie_ids is not None and len(seen_movie_ids) > 0:
        movie_df = movie_df[~movie_df["MovieID"].isin(seen_movie_ids)]

    top = movie_df.nlargest(top_n, "Score")
    return top[["MovieID", "Title", "Genres", "Score"]]

In [40]:
#top users - in top 5% by num_ratings
ratings_pre1980 = ratings[ratings["MovieID"].isin(movies_pre1980["MovieID"])]

user_counts = ratings_pre1980.groupby("UserID").size().reset_index(name="NumRatings")
threshold = user_counts["NumRatings"].quantile(0.95)
top_users = user_counts[user_counts["NumRatings"] >= threshold]

print(f"Top 5% threshold: {threshold} ratings")
print(f"Number of top users: {len(top_users)}")

np.random.seed(42)
top_user_id = np.random.choice(top_users["UserID"].values)
print(f"Selected top user: {top_user_id}")

Top 5% threshold: 140.39999999999964 ratings
Number of top users: 291
Selected top user: 2121


In [41]:
#info from these users -> profile embedding
top_user_ratings = ratings_pre1980[ratings_pre1980["UserID"] == top_user_id]
top_user_info = users[users["UserID"] == top_user_id].iloc[0]

print(f"User {top_user_id}: Gender={top_user_info['Gender']}, Age={top_user_info['Age']}, "
      f"Occupation={top_user_info['Occupation']}, Zip={top_user_info['Zip']}")
print(f"Number of pre-1980 ratings: {len(top_user_ratings)}")

last_timestamp = top_user_ratings["Timestamp"].max()
last_interaction = datetime.fromtimestamp(last_timestamp).strftime("%Y-%m-%d %H:%M:%S")
print(f"Last interaction: {last_interaction}")

User 2121: Gender=M, Age=56, Occupation=13, Zip=02356
Number of pre-1980 ratings: 164
Last interaction: 2000-11-19 15:41:36


In [42]:
#top user embedding from rated movies
movie_id_to_idx = {mid: idx for idx, mid in enumerate(movies_pre1980["MovieID"].values)}

user_movie_ids = top_user_ratings["MovieID"].values
user_ratings_vals = top_user_ratings["Rating"].values

valid_indices = []
valid_weights = []
for mid, rating in zip(user_movie_ids, user_ratings_vals):
    if mid in movie_id_to_idx:
        valid_indices.append(movie_id_to_idx[mid])
        valid_weights.append(rating)

valid_weights = np.array(valid_weights, dtype=np.float32)
user_movie_embeds = embeddings_pre1980[valid_indices]
top_user_embedding = np.average(user_movie_embeds, axis=0, weights=valid_weights)

print(f"Built user embedding from {len(valid_indices)} rated movies.")

Built user embedding from 164 rated movies.


In [43]:
seen_ids = set(top_user_ratings["MovieID"].values)
top_user_recs = recommend_movies(top_user_embedding, embeddings_pre1980, movies_pre1980, seen_ids, top_n=5)

print(f"\nTop 5 recommendations for Top User {top_user_id}:")
print(top_user_recs.to_string(index=False))


Top 5 recommendations for Top User 2121:
 MovieID                       Title                 Genres    Score
    3229 Another Man's Poison (1952)            Crime|Drama 0.961837
     960 Angel on My Shoulder (1946)            Crime|Drama 0.960855
    1344            Cape Fear (1962)     Film-Noir|Thriller 0.959744
    1287              Ben-Hur (1959) Action|Adventure|Drama 0.959208
     659          Purple Noon (1960)         Crime|Thriller 0.958384


In [44]:
# Cold user - no history at all
avg_ratings = ratings_pre1980.groupby("MovieID").agg(
    AvgRating=("Rating", "mean"),
    NumRatings=("Rating", "count")
).reset_index()

popular = avg_ratings[avg_ratings["NumRatings"] >= 50].nlargest(5, "AvgRating")
cold_user_recs = movies_pre1980[movies_pre1980["MovieID"].isin(popular["MovieID"])].copy()
cold_user_recs = cold_user_recs.merge(popular, on="MovieID")

print("Top 5 recommendations for Cold User (no history):")
print(cold_user_recs[["MovieID", "Title", "Genres", "AvgRating", "NumRatings"]].to_string(index=False))

Top 5 recommendations for Cold User (no history):
 MovieID                                                               Title             Genres  AvgRating  NumRatings
     858                                               Godfather, The (1972) Action|Crime|Drama   4.524966        2223
     904                                                  Rear Window (1954)   Mystery|Thriller   4.476190        1050
     922                       Sunset Blvd. (a.k.a. Sunset Boulevard) (1950)          Film-Noir   4.491489         470
    2019 Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)       Action|Drama   4.560510         628
    2905                                                      Sanjuro (1962)   Action|Adventure   4.608696          69


In [45]:
#top and cold user json info
top_user_result = {
    "User_Type": "Top User",
    "UserID": int(top_user_id),
    "Gender": top_user_info["Gender"],
    "Age": int(top_user_info["Age"]),
    "Occupation": int(top_user_info["Occupation"]),
    "Zip": top_user_info["Zip"],
    "Num_Ratings_Pre1980": int(len(top_user_ratings)),
    "Last_Interaction_Time": last_interaction,
    "Recommendations": top_user_recs[["MovieID", "Title", "Genres", "Score"]].to_dict(orient="records")
}

cold_user_result = {
    "User_Type": "Cold User",
    "UserID": None,
    "Gender": None,
    "Age": None,
    "Occupation": None,
    "Zip": None,
    "Num_Ratings_Pre1980": 0,
    "Last_Interaction_Time": None,
    "Recommendations": cold_user_recs[["MovieID", "Title", "Genres", "AvgRating", "NumRatings"]].to_dict(orient="records")
}

results_pre1980 = {"dataset": "pre-1980", "users": [top_user_result, cold_user_result]}

with open("outputs/recommendations_pre1980.json", "w") as f:
    json.dump(results_pre1980, f, indent=2, default=str)

print("Saved recommendations_pre1980.json locally.")

Saved recommendations_pre1980.json locally.


In [46]:
upload_file_to_s3("outputs/recommendations_pre1980.json", BUCKET_NAME, "outputs/recommendations_pre1980.json")
print("Pre-1980 recommendations saved to S3.")

Uploading outputs/recommendations_pre1980.json -> s3://yegon-jay-hw2/outputs/recommendations_pre1980.json
Pre-1980 recommendations saved to S3.


In [47]:
#Task 4

In [48]:
#text for all movies
movies["Text"] = movies.apply(build_movie_text, axis=1)
print(f"Total movies: {len(movies)}")
movies[["Title", "Text"]].head(10)

Total movies: 3883


,Title,Text
0,Toy Story (1995),Toy Story 1995 Animation Children's Comedy
1,Jumanji (1995),Jumanji 1995 Adventure Children's Fantasy
2,Grumpier Old Men (1995),Grumpier Old Men 1995 Comedy Romance
3,Waiting to Exhale (1995),Waiting to Exhale 1995 Comedy Drama
4,Father of the Bride Part II (1995),Father of the Bride Part II 1995 Comedy
5,Heat (1995),Heat 1995 Action Crime Thriller
6,Sabrina (1995),Sabrina 1995 Comedy Romance
7,Tom and Huck (1995),Tom and Huck 1995 Adventure Children's
8,Sudden Death (1995),Sudden Death 1995 Action
9,GoldenEye (1995),GoldenEye 1995 Action Adventure Thriller


In [49]:
#embeddings for all movies
texts_all = movies["Text"].tolist()
embeddings_all = generate_embeddings(texts_all, tokenizer, model)
print(f"Full embeddings shape: {embeddings_all.shape}")

Processed 32/3883 movies
Processed 192/3883 movies
Processed 352/3883 movies
Processed 512/3883 movies
Processed 672/3883 movies
Processed 832/3883 movies
Processed 992/3883 movies
Processed 1152/3883 movies
Processed 1312/3883 movies
Processed 1472/3883 movies
Processed 1632/3883 movies
Processed 1792/3883 movies
Processed 1952/3883 movies
Processed 2112/3883 movies
Processed 2272/3883 movies
Processed 2432/3883 movies
Processed 2592/3883 movies
Processed 2752/3883 movies
Processed 2912/3883 movies
Processed 3072/3883 movies
Processed 3232/3883 movies
Processed 3392/3883 movies
Processed 3552/3883 movies
Processed 3712/3883 movies
Processed 3872/3883 movies
Full embeddings shape: (3883, 768)


In [50]:
#saving full embeddings and movie list to s3
np.save("outputs/embeddings_full.npy", embeddings_all)
movies[["MovieID", "Title", "Genres", "Year", "Text"]].to_csv("outputs/movies_full.csv", index=False)

upload_file_to_s3("outputs/embeddings_full.npy", BUCKET_NAME, "outputs/embeddings_full.npy")
upload_file_to_s3("outputs/movies_full.csv", BUCKET_NAME, "outputs/movies_full.csv")
print("Full dataset embeddings saved to S3.")

Uploading outputs/embeddings_full.npy -> s3://yegon-jay-hw2/outputs/embeddings_full.npy
Uploading outputs/movies_full.csv -> s3://yegon-jay-hw2/outputs/movies_full.csv
Full dataset embeddings saved to S3.


In [51]:
#top user on full dataset
user_counts_full = ratings.groupby("UserID").size().reset_index(name="NumRatings")
threshold_full = user_counts_full["NumRatings"].quantile(0.95)
top_users_full = user_counts_full[user_counts_full["NumRatings"] >= threshold_full]

print(f"Top 5% threshold (full): {threshold_full} ratings")
print(f"Number of top users: {len(top_users_full)}")

np.random.seed(42)
top_user_id_full = np.random.choice(top_users_full["UserID"].values)
print(f"Selected top user: {top_user_id_full}")

Top 5% threshold (full): 556.0 ratings
Number of top users: 303
Selected top user: 1764


In [52]:
#top user info and embedding on full data
top_user_ratings_full = ratings[ratings["UserID"] == top_user_id_full]
top_user_info_full = users[users["UserID"] == top_user_id_full].iloc[0]

print(f"User {top_user_id_full}: Gender={top_user_info_full['Gender']}, Age={top_user_info_full['Age']}, "
      f"Occupation={top_user_info_full['Occupation']}, Zip={top_user_info_full['Zip']}")
print(f"Total ratings: {len(top_user_ratings_full)}")

last_ts_full = top_user_ratings_full["Timestamp"].max()
last_interaction_full = datetime.fromtimestamp(last_ts_full).strftime("%Y-%m-%d %H:%M:%S")
print(f"Last interaction: {last_interaction_full}")

movie_id_to_idx_full = {mid: idx for idx, mid in enumerate(movies["MovieID"].values)}

valid_indices_full = []
valid_weights_full = []
for mid, r in zip(top_user_ratings_full["MovieID"].values, top_user_ratings_full["Rating"].values):
    if mid in movie_id_to_idx_full:
        valid_indices_full.append(movie_id_to_idx_full[mid])
        valid_weights_full.append(r)

valid_weights_full = np.array(valid_weights_full, dtype=np.float32)
user_embeds_full = embeddings_all[valid_indices_full]
top_user_embedding_full = np.average(user_embeds_full, axis=0, weights=valid_weights_full)

print(f"Built user embedding from {len(valid_indices_full)} rated movies.")

User 1764: Gender=M, Age=18, Occupation=0, Zip=75149
Total ratings: 613
Last interaction: 2000-11-21 05:50:04
Built user embedding from 613 rated movies.


In [53]:
#recommendations for top user on full data
seen_ids_full = set(top_user_ratings_full["MovieID"].values)
top_user_recs_full = recommend_movies(top_user_embedding_full, embeddings_all, movies, seen_ids_full, top_n=5)

print(f"\nTop 5 recommendations for Top User {top_user_id_full} (full dataset):")
print(top_user_recs_full.to_string(index=False))


Top 5 recommendations for Top User 1764 (full dataset):
 MovieID                                      Title                 Genres    Score
     330                 Tales from the Hood (1995)          Comedy|Horror 0.970086
     861                            Supercop (1992)        Action|Thriller 0.969078
    1608                       Air Force One (1997)        Action|Thriller 0.967305
    3662 Puppet Master III: Toulon's Revenge (1991) Horror|Sci-Fi|Thriller 0.966234
     370  Naked Gun 33 1/3: The Final Insult (1994)                 Comedy 0.965660


In [54]:
#cold user recommendations on full data
avg_ratings_full = ratings.groupby("MovieID").agg(
    AvgRating=("Rating", "mean"),
    NumRatings=("Rating", "count")
).reset_index()

popular_full = avg_ratings_full[avg_ratings_full["NumRatings"] >= 50].nlargest(5, "AvgRating")
cold_user_recs_full = movies[movies["MovieID"].isin(popular_full["MovieID"])].copy()
cold_user_recs_full = cold_user_recs_full.merge(popular_full, on="MovieID")

print("Top 5 recommendations for Cold User (full dataset):")
print(cold_user_recs_full[["MovieID", "Title", "Genres", "AvgRating", "NumRatings"]].to_string(index=False))

Top 5 recommendations for Cold User (full dataset):
 MovieID                                                               Title                    Genres  AvgRating  NumRatings
     318                                    Shawshank Redemption, The (1994)                     Drama   4.554558        2227
     745                                               Close Shave, A (1995) Animation|Comedy|Thriller   4.520548         657
     858                                               Godfather, The (1972)        Action|Crime|Drama   4.524966        2223
    2019 Seven Samurai (The Magnificent Seven) (Shichinin no samurai) (1954)              Action|Drama   4.560510         628
    2905                                                      Sanjuro (1962)          Action|Adventure   4.608696          69


In [55]:
#save results and upload to s3
top_user_result_full = {
    "User_Type": "Top User",
    "UserID": int(top_user_id_full),
    "Gender": top_user_info_full["Gender"],
    "Age": int(top_user_info_full["Age"]),
    "Occupation": int(top_user_info_full["Occupation"]),
    "Zip": top_user_info_full["Zip"],
    "Num_Ratings": int(len(top_user_ratings_full)),
    "Last_Interaction_Time": last_interaction_full,
    "Recommendations": top_user_recs_full[["MovieID", "Title", "Genres", "Score"]].to_dict(orient="records")
}

cold_user_result_full = {
    "User_Type": "Cold User",
    "UserID": None,
    "Gender": None,
    "Age": None,
    "Occupation": None,
    "Zip": None,
    "Num_Ratings": 0,
    "Last_Interaction_Time": None,
    "Recommendations": cold_user_recs_full[["MovieID", "Title", "Genres", "AvgRating", "NumRatings"]].to_dict(orient="records")
}

results_full = {"dataset": "full", "users": [top_user_result_full, cold_user_result_full]}

with open("outputs/recommendations_full.json", "w") as f:
    json.dump(results_full, f, indent=2, default=str)

upload_file_to_s3("outputs/recommendations_full.json", BUCKET_NAME, "outputs/recommendations_full.json")
print("Full dataset recommendations saved to S3.")

Uploading outputs/recommendations_full.json -> s3://yegon-jay-hw2/outputs/recommendations_full.json
Full dataset recommendations saved to S3.


In [56]:
#Task 5

In [58]:
#50 movies to rate, i'll pick 10 from them
sample_movies = movies.sample(50, random_state=1)[["MovieID", "Title", "Genres"]]
print(sample_movies.to_string(index=False))

 MovieID                                                                          Title                              Genres
    3658                                                  Quatermass and the Pit (1967)                              Sci-Fi
    2471                                                     Crocodile Dundee II (1988)                    Adventure|Comedy
    1076                                                          Innocents, The (1961)                            Thriller
    1103                                                   Rebel Without a Cause (1955)                               Drama
    1003                                                        Extreme Measures (1996)                      Drama|Thriller
    1398                                                         In Love and War (1996)                         Romance|War
    3304                                                             Blue Collar (1978)                         Crime|Drama
     819

In [59]:
#my own ratings
jay_ratings = {
    "MovieID": [2947, 3948, 1923, 457, 2997, 1172, 1256, 1288, 3686, 1590],
    "Title": [
        "Goldfinger (1964)",
        "Meet the Parents (2000)",
        "There's Something About Mary (1998)",
        "Fugitive, The (1993)",
        "Being John Malkovich (1999)",
        "Cinema Paradiso (1988)",
        "Duck Soup (1933)",
        "This Is Spinal Tap (1984)",
        "Flatliners (1990)",
        "Event Horizon (1997)"
    ],
    "Rating": [5, 4, 4, 5, 3, 4, 3, 4, 3, 2]
}

jay_profile = pd.DataFrame(jay_ratings)
print("Jay's movie ratings:")
print(jay_profile.to_string(index=False))

Jay's movie ratings:
 MovieID                               Title  Rating
    2947                   Goldfinger (1964)       5
    3948             Meet the Parents (2000)       4
    1923 There's Something About Mary (1998)       4
     457                Fugitive, The (1993)       5
    2997         Being John Malkovich (1999)       3
    1172              Cinema Paradiso (1988)       4
    1256                    Duck Soup (1933)       3
    1288           This Is Spinal Tap (1984)       4
    3686                   Flatliners (1990)       3
    1590                Event Horizon (1997)       2


In [60]:
#user profile to s3
jay_profile_dict = {
    "UserID": "jay",
    "Name": "Jay",
    "Ratings": jay_profile.to_dict(orient="records"),
    "Num_Ratings": len(jay_profile),
    "Avg_Rating": round(jay_profile["Rating"].mean(), 2),
    "Created": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
}

with open("outputs/jay_profile.json", "w") as f:
    json.dump(jay_profile_dict, f, indent=2)

upload_file_to_s3("outputs/jay_profile.json", BUCKET_NAME, "outputs/jay_profile.json")
print("User profile saved to S3.")

Uploading outputs/jay_profile.json -> s3://yegon-jay-hw2/outputs/jay_profile.json
User profile saved to S3.


In [61]:
#my preference embedding
movie_id_to_idx_full = {mid: idx for idx, mid in enumerate(movies["MovieID"].values)}

jay_indices = []
jay_weights = []
for _, row in jay_profile.iterrows():
    mid = row["MovieID"]
    if mid in movie_id_to_idx_full:
        jay_indices.append(movie_id_to_idx_full[mid])
        jay_weights.append(row["Rating"])

jay_weights = np.array(jay_weights, dtype=np.float32)
jay_embeds = embeddings_all[jay_indices]
jay_embedding = np.average(jay_embeds, axis=0, weights=jay_weights)

print(f"Built Jay's embedding from {len(jay_indices)} rated movies.")

Built Jay's embedding from 10 rated movies.


In [62]:
#my 5 recommendations
jay_seen = set(jay_profile["MovieID"].values)
jay_recs = recommend_movies(jay_embedding, embeddings_all, movies, jay_seen, top_n=5)

print("Top 5 recommendations for Jay:")
print(jay_recs.to_string(index=False))

Top 5 recommendations for Jay:
 MovieID                      Title          Genres    Score
     449 Fear of a Black Hat (1993)          Comedy 0.969892
    2786   Haunted Honeymoon (1986)          Comedy 0.967364
    2249      My Blue Heaven (1990)          Comedy 0.966397
    2552 My Boyfriend's Back (1993)          Comedy 0.966150
     861            Supercop (1992) Action|Thriller 0.965779


In [63]:
#recommendations -> s3
jay_result = {
    "User_Type": "Custom (Jay)",
    "UserID": "jay",
    "Num_Ratings": len(jay_profile),
    "Avg_Rating": round(jay_profile["Rating"].mean(), 2),
    "Rated_Movies": jay_profile[["MovieID", "Title", "Rating"]].to_dict(orient="records"),
    "Recommendations": jay_recs[["MovieID", "Title", "Genres", "Score"]].to_dict(orient="records")
}

with open("outputs/jay_recommendations.json", "w") as f:
    json.dump(jay_result, f, indent=2, default=str)

upload_file_to_s3("outputs/jay_recommendations.json", BUCKET_NAME, "outputs/jay_recommendations.json")
print("Jay's recommendations saved to S3.")

Uploading outputs/jay_recommendations.json -> s3://yegon-jay-hw2/outputs/jay_recommendations.json
Jay's recommendations saved to S3.


In [64]:
#all files in s3?
print("All files on S3:")
response = s3.list_objects_v2(Bucket=BUCKET_NAME)
for obj in response.get("Contents", []):
    print(f"  {obj['Key']}  ({obj['Size']} bytes)")

All files on S3:
  ml-1m/README  (5577 bytes)
  ml-1m/movies.dat  (171308 bytes)
  ml-1m/ratings.dat  (24594131 bytes)
  ml-1m/users.dat  (134368 bytes)
  outputs/embeddings_full.npy  (11928704 bytes)
  outputs/embeddings_pre1980.npy  (2724992 bytes)
  outputs/jay_profile.json  (1086 bytes)
  outputs/jay_recommendations.json  (1789 bytes)
  outputs/movies_full.csv  (324540 bytes)
  outputs/movies_pre1980.csv  (76262 bytes)
  outputs/recommendations_full.json  (2480 bytes)
  outputs/recommendations_pre1980.json  (2471 bytes)
